# 04 — Rate reconciliation + κ-floor screen (Step 4)

The node that made the rate set trustworthy: reconciling pynucastro's graph
against MESA r23.05.1's softwired networks (ADR 0003), measuring rate VALUES
against the MESA probe, screening for the Appendix-B (gh-575) inverse-rate
bug, and screening κ at NSE for a spurious floor. This node produced the
project's **blocking construction gate**: reverse rates must be
`DerivedRate(use_pf=True)` or MESA-side — raw JINA v-flag reverses are
FORBIDDEN at T9 ≥ 3.

Exploratory only — all figures re-plot cached CSVs under `data/mesa_cache/`;
citable values are the RESULTS.md 2026-07-09/10 rows.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nbsupport as nbs

nbs.style()
CACHE = nbs.REPO / "data" / "mesa_cache"
NETS = ["mesa_80", "mesa_151"]

In [ ]:
nbs.provenance_header(
    "04",
    "Rate reconciliation + κ-floor screen",
    nbs.status_of(
        [],
        [
            "Softwired inverse-rate equilibrium",
            "Which weak-rate tables",
            "pynucastro TabularRate precedence",
        ],
    ),
    results_rows=[
        "2026-07-09: reconciliation — MESA_ONLY = 0, 607/1518 reactions, DEFAULT_TABULAR_ORDERING (ADR 0003)",
        "2026-07-10: rate cross-check — T9=5 weak node median |Δlog10| = 0.003; off-node 0.03–0.18 dex (MESA-bilinear vs pynucastro interpolant)",
        "2026-07-09: Appendix-B — stock r23.05.1 HAS the gh-575 bug: |Δlog10| = 10.0–11.1 (|ΔN|=1) and 20.6–22.7 (|ΔN|=2), tracking |ΔN|·log10(fac·T9^{3/2}) within ≲0.35 dex; BEYOND the paper: chapter-8 1→3 reverses (incl. the triple-α reverse) low by 9.5–11.3 dex (commit b8665c0)",
        "2026-07-10: MESA 24.08.1 installed side-by-side and the gh-575 fix verified (residual ≤ 1.9 dex; commit 156c73f)",
        "2026-07-10: κ-floor screen — raw v-flag median κ 6.6e-2 / 1.3e-1, suspects 268/280 and 656/672; stock-MESA residual floor 3.6e-3 / 3.3e-3",
    ],
    data=[
        "data/mesa_cache/crosscheck_{bare,screen,weak}_mesa_{80,151}.csv",
        "data/mesa_cache/appendixb_comparison.csv",
        "data/mesa_cache/kappa_nse_mesa_{80,151}.csv",
    ],
    scripts=[
        "scripts/reconcile_reactions.py",
        "scripts/crosscheck_rates.py",
        "scripts/appendixb_check.py",
        "scripts/kappa_floor_screen.py",
    ],
)

## Figure 1 — rate-value agreement by category

`dlog10 = log10(pyna) − log10(MESA)` per reaction per T9, over the regime
box. Categories come from the ADR-0003 disposition: `reaclib_forward` (the
bulk), `db_inverse` (detailed-balance reverses), `construction_swap`,
`appendixb_excluded` (channels quarantined by the gh-575 screen), and the
weak sector.

In [ ]:
bare = pd.concat([pd.read_csv(CACHE / f"crosscheck_bare_{n}.csv").assign(network=n) for n in NETS])
weak = pd.concat([pd.read_csv(CACHE / f"crosscheck_weak_{n}.csv").assign(network=n) for n in NETS])
allc = pd.concat([bare, weak], ignore_index=True)
cats = [c for c in allc.category.value_counts().index]

fig, ax = plt.subplots(figsize=(10, 4.6))
data = [allc.loc[allc.category == c, "dlog10"].dropna().to_numpy() for c in cats]
bp = ax.boxplot(data, tick_labels=cats, showfliers=True, whis=(1, 99),
                flierprops=dict(marker=".", markersize=2, alpha=0.35))
ax.axhspan(-0.004, 0.004, color="#009E73", alpha=0.18, label="forward band ±0.004 dex")
ax.axhline(0, color="#333333", lw=0.8)
ax.set_ylabel("Δlog10 (pynucastro − MESA)")
ax.set_yscale("symlog", linthresh=1e-3)
ax.set_title("Rate-value agreement vs MESA r23.05.1 probe, by disposition category (both nets)")
plt.setp(ax.get_xticklabels(), rotation=18, ha="right")
ax.legend(loc="upper right")
nbs.caption(
    fig,
    "Forward REACLIB rates agree to numerical noise; the weak sector's spread is the "
    "MESA-bilinear vs pynucastro-interpolant difference on the SAME tables (median |Δlog10| "
    "0.003 at the T9=5 node, 0.03–0.18 dex off-node, β⁻ tail ≤ 0.8 dex) — the training labels "
    "contain MESA's bilinear values. appendixb_excluded is the quarantined gh-575 set (fig. 2).",
    results=[
        "RESULTS.md 2026-07-10 rate cross-check rows (docs/rate-crosscheck.md)",
        "RESULTS.md 2026-07-09 reconciliation rows (ADR 0003)",
    ],
    scripts=["scripts/crosscheck_rates.py"],
)

## Figure 2 — the Appendix-B (gh-575) signature

MESA r23.05.1 computes multi-particle inverse rates from detailed balance
while omitting the **thermal phase-space factor** — `fac · T9^{3/2}` per
|ΔN|, with fac = (1e9 k_B / 2πℏ²N_A)^{3/2}/N_A (density-INDEPENDENT). It is
applied only when the forward has a single product, so every reverse of a
chapter with Nout ≠ 1 and Nin ≠ Nout loses it. `expected_missing_log10` is
the displacement that mechanism predicts (|ΔN|·log10(fac·T9^{3/2})); `dlog10`
is the measured pyna−MESA gap. Agreement between the two is what makes the
attribution **dispositive** rather than circumstantial.

In [ ]:
ab = pd.read_csv(CACHE / "appendixb_comparison.csv")
sus = ab[ab["class"] == "multi_body_inverse"]
ctl = ab[ab["class"] == "control_ch4_reverse"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), width_ratios=[2, 1])
ax = axes[0]
g = sus.groupby("mesa_name").agg(meas=("dlog10", "median"), pred=("expected_missing_log10", "median"))
g = g.sort_values("meas")
y = np.arange(len(g))
ax.barh(y - 0.2, g["meas"], 0.4, label="measured Δlog10 (pyna − MESA)")
ax.barh(y + 0.2, g["pred"], 0.4, label="predicted by missing |ΔN|·log10(fac·T9$^{3/2}$)")
ax.set_yticks(y, g.index, fontsize=7)
ax.set_xlabel("dex")
ax.set_title("Multi-body inverse channels: measured vs predicted displacement")
ax.legend(loc="lower right")

ax = axes[1]
for label, d, c in [("multi-body inverse", sus.dlog10, "#D55E00"),
                    ("chapter-4 reverse (control)", ctl.dlog10, "#0072B2")]:
    ax.hist(d.dropna(), bins=25, alpha=0.65, label=label, color=c)
ax.set_xlabel("Δlog10 (pyna − MESA)")
ax.set_ylabel("channels × T9")
ax.set_title("Bug class vs control")
ax.legend(fontsize=8)
nbs.caption(
    fig,
    "The suspect channels' displacement tracks the predicted missing phase-space term (within "
    "≲0.35 dex) while "
    "the chapter-4 reverse controls sit near zero — stock r23.05.1 HAS the gh-575 bug (10.0–22.7 "
    "dex on 9/11 multi-body inverse channels); MESA 24.08.1 fixes it (residual ≤ 1.9 dex). This is "
    "the same bug that later turned out to sit INSIDE the shipped training labels (notebook 09).",
    results=[
        "RESULTS.md 2026-07-09 Appendix-B rows (scripts/appendixb_check.py, commit b8665c0)",
        "RESULTS.md 2026-07-10 24.08.1 fix-verification row (commit 156c73f)",
    ],
    scripts=["scripts/appendixb_check.py"],
)

## Figure 3 — the κ floor at NSE: why raw v-flag reverses are forbidden

κ_r = |f⁺−f⁻|/(f⁺+f⁻) at a true NSE composition **must** collapse toward 0
for strong pairs (that is what detailed balance means). Three rate sources
at the same NSE states, T9 ∈ {5.0, 6.3, 7.9}:

- `kappa_pyna` — the graph as built with **raw JINA v-flag reverses**
  (pf-free fits) ⇒ manufactures a spurious floor,
- `kappa_mesa` — stock r23.05.1 ratios,
- `kappa_mesa24` — the gh-575-fixed 24.08.1 ratios.

The pf-true (`DerivedRate(use_pf=True)`) reverses collapse to 1e-13…1e-11 —
that measurement is a RESULTS.md row, not a column of this CSV, so it is
annotated rather than plotted.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, net in zip(axes, NETS):
    d = pd.read_csv(CACHE / f"kappa_nse_{net}.csv")
    floors = d.groupby(["fwd", "rev"])[["kappa_pyna", "kappa_mesa", "kappa_mesa24"]].min()
    bins = np.logspace(-14, 0, 45)
    for col, label, c in [
        ("kappa_pyna", "pyna raw v-flag (FORBIDDEN)", "#D55E00"),
        ("kappa_mesa", "MESA stock r23.05.1", "#0072B2"),
        ("kappa_mesa24", "MESA 24.08.1 (fixed)", "#009E73"),
    ]:
        ax.hist(np.clip(floors[col].dropna(), 1e-14, 1), bins=bins, alpha=0.55,
                label=label, color=c)
    ax.axvline(1e-3, color="#333333", ls="--", lw=1)
    ax.text(1.1e-3, ax.get_ylim()[1] * 0.9, "suspect > 1e-3", fontsize=8, rotation=90, va="top")
    ax.axvspan(1e-13, 1e-11, color="#009E73", alpha=0.15)
    ax.text(3e-13, ax.get_ylim()[1] * 0.55, "pf-true reverses\ncollapse here\n(RESULTS 2026-07-10)",
            fontsize=7.5, ha="center", color="#005f45")
    ax.set_xscale("log")
    ax.set_xlabel("per-pair κ floor at NSE (min over states)")
    ax.set_title(net)
axes[0].set_ylabel("strong/EM pairs")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("κ floor at NSE by rate source — UNSCREENED κ (detailed-balance diagnostic)", y=1.02)
nbs.caption(
    fig,
    "Raw v-flag reverses manufacture a floor at median κ 6.6e-2 / 1.3e-1 (suspects 268/280 and "
    "656/672 pairs > 1e-3, mesa_80/151) — a κ of 0.066 at NSE is physically impossible and would "
    "have made every reaction look 'active' in the kill-test. Attribution to pf-free fits is "
    "dispositive: pf-true reverses collapse to 1e-13…1e-11. Stock MESA's own residual floor is "
    "3.6e-3 / 3.3e-3. Consequence: the blocking κ/flux construction gate in CLAUDE.md.",
    results=["RESULTS.md 2026-07-10 κ-floor screen rows (scripts/kappa_floor_screen.py, commit f66328a)"],
    scripts=["scripts/kappa_floor_screen.py"],
)

## What this notebook does NOT show

- The **screened**-κ offset at NSE (~7e-2, a real config property, not a pf
  artifact): notebook 05, kept in a separate figure per the two-κ rule.
- The weak-table precedence measurement (LMP > Oda > FFN,
  `use_suzuki_weak_rates=.false.`) and off-grid clip-vs-extrapolate
  behavior — `docs/rate-crosscheck.md`.
- That the **shipped labels themselves** carry gh-575: that is the Step-6
  finding in notebook 09.